# TorchLens UI Sandbox

Interactive notebook for tinkering with torchlens during development.
Run cells, tweak parameters, inspect outputs. Not a test — a workbench.

In [ ]:
import sys
import os

# Ensure torchlens and test models are importable
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

import torch
import torch.nn as nn
from IPython.display import Image, display
from torchlens import log_forward_pass, validate_forward_pass, show_model_graph
import example_models

print(f"torch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

---
## 1. Basic Logging

In [ ]:
model = example_models.NestedModules()
x = torch.rand(5, 5)
log = log_forward_pass(model, x, random_seed=42)
log

In [ ]:
# str() vs repr()
print("=== str ===")
print(str(log))
print("\n=== repr ===")
print(repr(log))

---
## 2. Layer Access

In [ ]:
# Access by index
log[0]

In [ ]:
# Access by name
print(list(log.layer_labels)[:10])
log[log.layer_labels[0]]

In [ ]:
# LayerLog aggregates (per-layer, across passes)
log.layers[0]

---
## 3. Module Access

In [ ]:
# Module accessor
log.modules

In [ ]:
log.modules["self"]

In [ ]:
log.modules[0]

---
## 4. Parameters

In [ ]:
log.params

In [ ]:
log.params[0]

---
## 5. DataFrames

In [ ]:
log.to_pandas()

In [ ]:
log.modules.to_pandas()

---
## 6. Visualization

In [ ]:
# Unrolled graph
show_model_graph(
    model,
    x,
    vis_opt="unrolled",
    vis_nesting_depth=2,
    vis_outpath="test_outputs/graphs/sandbox_unrolled",
    save_only=True,
)
display(Image(filename="test_outputs/graphs/sandbox_unrolled.gv.png"))

In [ ]:
# Rolled graph
show_model_graph(
    model,
    x,
    vis_opt="rolled",
    vis_nesting_depth=2,
    vis_outpath="test_outputs/graphs/sandbox_rolled",
    save_only=True,
)
display(Image(filename="test_outputs/graphs/sandbox_rolled.gv.png"))

---
## 7. Recurrent Model

In [ ]:
rec_model = example_models.RecurrentParamsSimple()
rec_log = log_forward_pass(rec_model, torch.rand(5, 5), random_seed=42)
rec_log

In [ ]:
# Multi-pass layers
for lbl, layer in rec_log.layer_logs.items():
    if hasattr(layer, "num_passes") and layer.num_passes > 1:
        print(f"{lbl}: {layer.num_passes} passes")

---
## 8. Buffers

In [ ]:
buf_model = example_models.BufferModel()
buf_log = log_forward_pass(buf_model, torch.rand(12, 12), random_seed=42)
buf_log.buffers

---
## 9. save_new_activations

In [ ]:
simple = example_models.SimpleBranching()
x_branch = torch.rand(6, 3, 224, 224)
slog = log_forward_pass(simple, x_branch, random_seed=42)
print(f"Before: output = {slog[-1].tensor_contents.sum():.4f}")

slog.save_new_activations(simple, x_branch.clone(), random_seed=42)
print(f"After:  output = {slog[-1].tensor_contents.sum():.4f}")

---
## 10. Validation

In [ ]:
result = validate_forward_pass(model, x, random_seed=42)
print(f"Validation passed: {result}")

---
## 11. Real-World Model (optional)

In [ ]:
try:
    import torchvision

    rn18 = torchvision.models.resnet18()
    rn18.eval()
    rn_log = log_forward_pass(rn18, torch.rand(1, 3, 224, 224), random_seed=42)
    rn_log
except ImportError:
    print("torchvision not installed — skipping")

In [ ]:
# Scratch space — try things here
